# Group 25 — Arrival-Time Prediction of EV Charging Session Energy with Completed-Session History

**New package assembled 06 November 2026 from the newly supplied ChargingRecords.xlsx and notebook.**
This notebook retains the supplied research workflow and adds the balanced/nested checks needed
for the report. Historical Python estimator code, original CSV, fitted models and saved experiment
outputs are reused from the earlier research package; they are identified in PACKAGE_INFO.json.

- Predict final session Demand (kWh) at arrival.
- RQ1: charging patterns; RQ2: model/pipeline comparisons; RQ3: identity/history ablation.
- Primary model: equal CatBoost blend; online completed-label feedback, fixed weights.
- The final period was examined previously. Results remain exploratory.

**Colab:** Runtime > Run all, upload ADY_Group25_Colab_Package_New.zip when prompted.
Keep all five RUN flags False for cached-model evaluation and verification. This is not a claim
that the models have been trained again. The standalone notebook and ZIP must be from this package.
Use README_COLAB_VI.md and REPRODUCE.md; the current Overleaf manuscript is not bundled as an old PDF.

## 1. Setup and execution modes

CPU is supported. Dependencies used by the notebook, workbook audit, figure export and
checkpoint are pinned. Restart runtime and Run all if the setup cell asks for a restart.
All five flags default to False. Keep them False to verify existing predictions/models.

| Flag | True requests |
|---|---|
| RUN_TRAIN | Refit original study, then related diagnostics |
| RUN_REVISION_CHECKS | Recompute baseline, ablation, reporting-delay and uncertainty checks |
| RUN_FEEDBACK_CHECKS | Repeat fixed common-input comparison |
| RUN_BALANCED_SEARCH | Repeat 185-fit selection diagnostic |
| RUN_NESTED_TEMPORAL | Repeat 595-fit nested temporal experiment |

The default run recomputes metrics, features, SQL, figures and inference tests without
retraining. Log warnings and errors are retained; no verifier is skipped to conceal a mismatch.

In [ ]:
from pathlib import Path
import sys, os, importlib.util, importlib.metadata, subprocess, json, hashlib, shutil, platform
required_versions = [('numpy', 'numpy', '2.3.5'), ('pandas', 'pandas', '2.2.3'), ('scikit-learn', 'sklearn', '1.8.0'), ('scipy', 'scipy', '1.17.0'), ('matplotlib', 'matplotlib', '3.10.8'), ('seaborn', 'seaborn', '0.13.2'), ('joblib', 'joblib', '1.5.3'), ('threadpoolctl', 'threadpoolctl', '3.6.0'), ('lightgbm', 'lightgbm', '4.7.0'), ('catboost', 'catboost', '1.2.10'), ('narwhals', 'narwhals', '2.27.1'), ('graphviz', 'graphviz', '0.21'), ('ipython', 'IPython', '9.17.1'), ('openpyxl', 'openpyxl', '3.1.5'), ('Pillow', 'PIL', '11.3.0'), ('reportlab', 'reportlab', '4.4.9'), ('pypdfium2', 'pypdfium2', '4.30.0'), ('nbformat', 'nbformat', '5.10.4'), ('matplotlib-inline', 'matplotlib_inline', '0.1.7')]
needed=[]
for distribution,module,version in required_versions:
    try: installed=importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: needed.append((distribution,module,version))
if needed:
    loaded=any(module in sys.modules for _,module,_ in needed)
    subprocess.check_call([sys.executable,'-m','pip','install']+[f'{name}=={version}' for name,_,version in needed])
    if loaded:
        raise RuntimeError('Dependencies were updated after import. Restart runtime, then Run all again.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display,Markdown
SEED=42
RUN_TRAIN=False
RUN_REVISION_CHECKS=False
RUN_FEEDBACK_CHECKS=False
RUN_BALANCED_SEARCH=False
RUN_NESTED_TEMPORAL=False
np.random.seed(SEED)
pd.set_option('display.max_columns',50)
pd.set_option('display.float_format',lambda value:f'{value:,.3f}')
sns.set_theme(style='whitegrid',context='notebook')
BLUE,LIGHT_BLUE,RED='#2F6690','#7FB3D5','#B23A48'
print('Python:',platform.python_version(),'| Cached verification mode:',not any([RUN_TRAIN,RUN_REVISION_CHECKS,RUN_FEEDBACK_CHECKS,RUN_BALANCED_SEARCH,RUN_NESTED_TEMPORAL]))

## 2. Load this package and check the new workbook against the canonical CSV

Upload ADY_Group25_Colab_Package_New.zip when prompted. Only a package with the new
PACKAGE_INFO.json identifier is selected; older extracted project folders are ignored.
The notebook imports project modules from the extracted ZIP. A notebook plus XLSX alone
does not contain those modules or checkpoint.

data/ChargingRecords.xlsx is the workbook supplied for this rebuild. It is compared
row by row, after date/time normalization, to data/ChargingRecords_original.csv.
The original CSV byte representation is retained to preserve the saved model's audit hashes.

In [ ]:
import zipfile
PACKAGE_ID='group25-rebuild-2026-11-06'
REQUIRED=['arrival_history.py', 'train_arrival.py', 'study_plots.py', 'predict_arrival.py', 'importance_diagnostics.py', 'revision_checks.py', 'report_alignment.py', 'feedback_checks.py', 'verify_feedback.py', 'verify_results.py', 'balanced_search.py', 'verify_balanced_search.py', 'nested_temporal.py', 'verify_nested_temporal.py', 'verify_inference_ids.py', 'verify_inference_inputs.py', 'data/ChargingRecords_original.csv', 'data/ChargingRecords.xlsx', 'analysis_outputs/models/arrival_model_bundle.joblib', 'analysis_outputs/nested_temporal/completed.json', 'analysis_outputs/balanced_search/protocol.json', 'report_source/render_review_figures.py']
def locate_project(base):
    candidates=[base,base/'ADY_Group25',base/'group25_uploaded'/'ADY_Group25']
    if str(base)!='/content':
        candidates += [Path('/content/ADY_Group25'),Path('/content/group25_uploaded/ADY_Group25')]
    for p in candidates:
        marker=p/'PACKAGE_INFO.json'
        if marker.is_file() and all((p/name).is_file() for name in REQUIRED):
            if json.loads(marker.read_text(encoding='utf-8')).get('package_id')==PACKAGE_ID:return p.resolve()
    return None
root=locate_project(Path.cwd())
try: in_colab=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError: in_colab=False
if root is None and in_colab:
    from google.colab import files
    print('Upload ADY_Group25_Colab_Package_New.zip.')
    uploaded=files.upload()
    base=Path.cwd()/'group25_uploaded';base.mkdir(exist_ok=True)
    for filename in uploaded:
        if zipfile.is_zipfile(filename):
            with zipfile.ZipFile(filename) as archive:
                for member in archive.namelist():
                    if not (base/member).resolve().is_relative_to(base.resolve()):raise ValueError('Unsafe ZIP path')
                archive.extractall(base)
    root=locate_project(Path.cwd())
if root is None:
    raise FileNotFoundError('Extract the NEW ZIP, open its notebook and Run all. The full project folder must contain PACKAGE_INFO.json.')
os.chdir(root);sys.path.insert(0,str(root))
OUT=Path('analysis_outputs');OUT.mkdir(exist_ok=True)
REV=OUT/'revision'
check=subprocess.run([sys.executable,'compare_uploaded_sources.py'],check=True,capture_output=True,text=True)
print(check.stdout)
from arrival_history import (load_and_audit, chronological_split, build_features,
                             audit_feature_causality, CAT, CALENDAR, HISTORY, STAT_NAMES)
from train_arrival import run_study
from study_plots import sql_analysis, eda, model_figures
from revision_checks import run_revision_checks, FILES as REVISION_FILES, sha256, rounded_report_checks
from report_alignment import report_tables, report_figures, primary_report_checks, verify_revision_outputs
df, quarantine, audit = load_and_audit('data/ChargingRecords_original.csv', 'data')
expected_sha256 = 'd541259bed55e2a9bc3413756954e7a059b50f9c9031f1dfc6ec87cbf3afd358'
if audit['source_sha256'] != expected_sha256:
    raise ValueError('Raw snapshot differs. Do not reuse the report scores for changed input data.')
assert audit['raw_rows'] == 72856 and len(df) == 69570 and len(quarantine) == 3286
display(pd.DataFrame([audit]).drop(columns=['overlapping_rule_counts', 'columns_actually_supplied']))
display(df.head(3))

## 3. Data understanding and preparation — report Table 1

Source: Baek, Lee and Kim (2024), Scientific Data, DOI **10.1038/s41597-024-02942-9**;
Figshare **10.6084/m9.figshare.22495141.v1**. `Demand` is **kWh**; `Duration` is connection
**minutes**. Recorded Korean civil times are used. The released CSV has 13 fields;
capacity and AC/DC fields absent from it are not reconstructed.

The union of six overlapping quality flags quarantines 3,286 rows. The two-minute
duration/timestamp tolerance is a study choice. Valid high-energy sessions remain;
there is no energy quantile trimming, target cap or synthetic augmentation.


In [ ]:
quality = pd.DataFrame(audit['overlapping_rule_counts'].items(), columns=['quality_rule', 'rows'])
display(quality)
display(df.Demand.describe().to_frame('kWh'))
print('Raw:', audit['raw_rows'], '| Clean:', len(df), '| Quarantined:', len(quarantine))
print('Identified:', int(df.known_user.sum()), '| Anonymous:', int(df.known_user.eq(0).sum()))
print('Retained:', f"{100 * len(df) / audit['raw_rows']:.1f}%", '| Chargers:', df.ChargerID.nunique())
assert len(df) + len(quarantine) == audit['raw_rows']
assert int(df.known_user.sum()) == 40790 and df.ChargerID.nunique() == 2115
assert audit_feature_causality(df)


## 4. SQLite database

One row is one charging session. A unique charger catalog prevents joins from
multiplying sessions. These are retrospective cohort summaries, not measured
instantaneous power or causal effects.


In [ ]:
answers=sql_analysis(df,OUT)
print('PASS: charger join preserves rows and nine SQL results match pandas checks.')


## 5. SQL analysis

Nine queries cover scope, hour, weekday, month, chargers, identity availability,
weekday–hour hotspots, location and energy share. CTEs and window functions are
used for charger ranking and percentage shares.


In [ ]:
for name,answer in answers.items():
    print(name)
    display(answer if name == 'Q2_hour' else answer.head(15))


## 6. SQL and Python consistency checks

Agreement checks numerical calculations and join integrity. It does not independently
verify the original sensors. Energy is assigned to arrival cohorts, not delivery hours.


In [ ]:
print('PASS: Q1–Q9 checked, unique charger join, positive target and valid audited times.')
assert np.isclose(answers['Q9_share'].pct_energy.sum(),100)


## 7. Data visualization for RQ1

Descriptive SQL/EDA uses the whole cleaned snapshot, including test dates. This is disclosed;
these plots do not make the test period independent. Energy is attributed to the arrival cohort,
not physically delivered during that hour. September endpoints are partial months.

The original exploratory figures below are supplementary displays. Report Figure 2, with
seven-date block intervals for mean energy, is generated in Section 15 from computed results.


In [ ]:
eda(df,answers,OUT/'figures')


## 8. Feature engineering and information available at arrival

There are **137 predictors before encoding**: 12 calendar/availability fields, five categories
and six history groups × 20 statistics. `UserID=0` represents multiple anonymous customers.
It receives no pooled personal history. The `known_user` flag is constant in personalized models
and retained to share the all-user schema.

History admits a previous outcome only if **previous EndDatetime + reporting delay < new arrival**.
Current-session Demand, Duration, EndDatetime and outcome-derived ratios are excluded.
Historical duration is converted to **hours**. Historical energy per connection hour is a **kW
proxy averaged over the connection**, not instantaneous charging power. Gap-energy is a heuristic,
not measured SOC. See the complete suffix definitions exported by `revision_checks.py`.


In [ ]:
display(pd.DataFrame({'arrival_calendar': pd.Series(CALENDAR)}))
display(pd.DataFrame({'categorical': CAT}))
display(pd.DataFrame({'history_suffix': STAT_NAMES}))
print('History groups: global, charger, user, charger_weekday, user_weekday, user_charger')
print('Predictors before encoding:', len(CALENDAR) + len(CAT) + len(HISTORY))
assert len(CALENDAR) == 12 and len(CAT) == 5 and len(HISTORY) == 120
assert not {'Demand', 'Duration', 'EndDatetime'} & set(CALENDAR + CAT + HISTORY)


## 9. Fold-fitted preprocessing and temporal validation — report Table 2

Test is the last **20% of observed dates**, starting **19 July 2022**, rather than exactly 20%
of rows. Whole dates stay together. Unavailable fitting labels are purged. Five expanding folds
select by unweighted mean CV R². Imputation, StandardScaler, one-hot encoding and category
vocabularies are fitted inside each training fold. StandardScaler does not force values into [0,1].
The one-hot encoder uses `min_frequency=10`, `max_categories=64` per field and
`handle_unknown='infrequent_if_exist'`, which merges rare IDs and limits identity resolution.

Online feedback updates histories after completion and the buffer; **weights stay frozen**.
Frozen history admits only development-pool records during test. The final period was examined
in earlier experiments; neither current CV selection nor added diagnostics make it a blind test.


In [ ]:
dev, test, purged, cutoff = chronological_split(df)
display(pd.DataFrame([
    {'split': 'development context', 'rows': len(dev), 'identified_rows': int(dev.known_user.sum())},
    {'split': 'test context', 'rows': len(test), 'identified_rows': int(test.known_user.sum())},
    {'split': 'purged at boundary', 'rows': len(purged), 'identified_rows': int(purged.known_user.sum())}]))
assert len(dev) + len(test) + len(purged) == 69570
assert int(dev.known_user.sum()) + int(test.known_user.sum()) + int(purged.known_user.sum()) == 40790
assert set(dev.source_row).isdisjoint(test.source_row)
assert dev.label_available_at.max() < test.StartDatetime.min()
print('Test begins:', cutoff)
from train_arrival import temporal_folds
folds, fold_table = temporal_folds(df, dev)
display(fold_table)


## 10. Regression models, original selection and added diagnostic baseline

Seven initial families: Linear, Ridge, kNN, RF, HGB, LightGBM and CatBoost, plus Dummy Mean.
kNN fits at most 12,000 seeded rows; RF caps bootstrap draws at 30,000 per tree. Pipelines differ
in categorical representation, so their comparison is not a matched-encoding algorithm test.
All use a zero prediction floor and retain targets on the original kWh scale.

**User historical mean** is an added, unfitted diagnostic: use available personal mean energy,
otherwise available global mean. It follows the same completion rule. It does not enter or alter
the original selection. The report's added experiments are reproduced by `revision_checks.py`.


In [ ]:
if RUN_TRAIN:
    result, cv_table, test_table, protocol_table = run_study(df, OUT)
else:
    required_results = ['manifest.json', 'models/comparison_cv.csv',
        'models/personalized_test_results.csv', 'models/protocol_and_cohort_results.csv',
        'models/arrival_model_bundle.joblib', 'models/tuning_trials.csv',
        'models/feature_ablation.csv', 'models/selection_before_test.json',
        'models/block_bootstrap.json', 'models/personalized_test_predictions.csv']
    missing = [name for name in required_results if not (OUT / name).is_file()]
    if missing:
        raise FileNotFoundError('Missing original cached outputs. Upload the current package or set RUN_TRAIN=True: ' + ', '.join(missing))
    result = json.loads((OUT / 'manifest.json').read_text())
    protocol_table = pd.read_csv(OUT / 'models/protocol_and_cohort_results.csv')
    print('Loaded the original fitted study; no full retraining requested.')
if RUN_TRAIN or RUN_REVISION_CHECKS:
    revision = run_revision_checks('.', force=True)
else:
    missing = [name for name in REVISION_FILES if not (REV / name).is_file()]
    if missing:
        raise FileNotFoundError('Missing revised-report outputs. Upload ADY_Group25_Colab_Package.zip or set RUN_REVISION_CHECKS=True: ' + ', '.join(missing))
    cached = json.loads((REV / 'revision_manifest.json').read_text())
    expected_hashes = {'source_sha256': sha256('data/ChargingRecords_original.csv'),
                      'primary_model_sha256': sha256(OUT / 'models/arrival_model_bundle.joblib'),
                      'revision_code_sha256': sha256('revision_checks.py')}
    if any(cached.get(k) != v for k, v in expected_hashes.items()):
        raise RuntimeError('Revision cache belongs to changed data, model or code. Set RUN_REVISION_CHECKS=True to recompute.')
    revision = run_revision_checks('.')
cv_table, test_table = report_tables('.')
display(cv_table[['model', 'cv_R2', 'cv_R2_sd', 'cv_MAE', 'cv_RMSE']])
display(pd.read_csv(REV / 'user_mean_test.csv'))


## 11. Hyperparameter tuning and its limits

CatBoost and RF each receive six sampled configurations on the same five folds. Original
candidates remain eligible. All six actual CatBoost trials use 400 iterations; the nominal
700-iteration value is never sampled. The chosen depth, rate and regularization lie at boundaries.
This is a small randomized search, not an exhaustive search or proof of a global optimum.
Blend weights 0.25, 0.50 and 0.75 are compared by CV only. Selection is non-nested.


In [ ]:
trials = pd.read_csv(OUT / 'models/tuning_trials.csv')
display(trials)
cat_trials = trials[trials.family.eq('CatBoost')].parameters.map(json.loads)
print('Realized CatBoost iteration values:', sorted({p['iterations'] for p in cat_trials}))
display(pd.read_csv(REV / 'user_mean_folds.csv'))


## 12. Fixed-LightGBM feature ablation — report Table 5

The original three sets are supplemented with:

1. **All history, without raw UserID (136 columns):** still requires an account key to retrieve history.
2. **Nonpersonal history, without UserID (76 columns):** removes user, user–weekday and user–charger histories.

Both use the same initial LightGBM hyperparameters and five temporal folds. The comparison tests
representation under LightGBM, not the selected CatBoost blend. It does not establish usefulness
of every one of the 120 statistics or the gap-energy proxy.


In [ ]:
display(pd.read_csv(REV / 'feature_ablation_extended.csv'))
display(pd.read_csv(REV / 'history_statistics.csv'))
display(pd.read_csv(REV / 'feature_dictionary_complete.csv').head(20))
print('Fold-count reconciliation:')
print(json.dumps(json.loads((REV / 'fold_reconciliation.json').read_text()), indent=2))


## 13. Original model selection before test scoring

Mean fold R² is the declared selection criterion; it normalizes squared error relative to each
window's target variation. MAE measures absolute kWh error. Tuned RF has lower CV MAE than
the blend, so selection by R² does not imply minimum MAE. Neither R² nor MAE measures accuracy
as a classification percentage. The tiny gain over original CatBoost needs cautious interpretation.

The original equal blend of CatBoost and tuned CatBoost remains selected. Added baselines,
delay scenarios and test results do not change its components or weights.


In [ ]:
selection = json.loads((OUT / 'models/selection_before_test.json').read_text())
print('CV-selected:', selection['model'], '| CV R²:', selection['cv_R2'])
print('Components and weights:', list(zip(selection['blend_parts'], selection['blend_weights'])))
assert selection['test_errors_used_for_selection'] is False
print('History protocol: online label feedback; frozen weights within each evaluation window.')
print('Prior test exposure remains an explicit limitation.')


## 14. Chronological test, cohorts and reporting delay — report Tables 4, 6 and 7

The primary evaluation has **10,999 identified arrivals**. All-user routing applies the personalized
model to identified arrivals and a general LightGBM trained on all development users to anonymous
arrivals. The primary R² must not be described as the overall all-user score.

Delay checks use 0 s, 60 s, 5 min, 1 hour and 1 day. Histories are rebuilt and fitting labels
repurged at each delay; the two fixed CatBoost configurations are refitted. Test dates and blend
weights remain fixed. The 60-second case reuses the original verified model. These are sensitivity
checks, not a test-driven search for the best real-world delay.


In [ ]:
display(test_table[['model', 'R2', 'MAE', 'RMSE', 'MedAE']])
display(protocol_table)
display(pd.read_csv(REV / 'delay_sensitivity.csv'))
display(pd.read_csv(REV / 'paired_mae_bootstrap.csv'))
display(Markdown('Primary score: **identified users with online feedback**, not every session.'))


## 15. Report figures, residuals, permutation and uncertainty

Report Figure 2 uses 2,000 circular seven-observed-date bootstrap draws for pointwise hourly
mean-energy intervals. Report Figure 3 uses all 10,999 test sessions in two independently scaled
hexbin panels, with equal 0–100 kWh prediction axes and a residual median for 5 kWh bins with n ≥ 20.
Residual = actual − predicted; positive values denote underprediction. Nonnegative energy imposes
the bound residual ≥ −prediction.

Report Figure 4 uses **2,500 test sessions and 10 shuffles per group**. Bars show shuffle SD,
not population confidence intervals. Feature correlations and unrealistic shuffled combinations
limit causal interpretation. Paired MAE gains compare the selected blend to Dummy Mean, user
historical mean and original CatBoost using 2,000 seven-date block draws. Intervals remain conditional
on this observed window and model, not calibrated prediction intervals for individual arrivals.


The current report uses four numbered figures from `report_source/figures`; earlier full-study charts
remain supplementary. Matched-input scores are diagnostic and do not replace the primary selection.


In [ ]:
errors = model_figures(OUT)
report_figures('.')
display(pd.read_csv(REV / 'permutation_10.csv'))
for group in ['year_month', 'is_weekend', 'account_seen_in_training']:
    path = OUT / f'models/error_by_{group}.csv'
    if path.is_file():
        display(pd.read_csv(path))
uncertainty = json.loads((OUT / 'models/block_bootstrap.json').read_text())
print(json.dumps(uncertainty, indent=2))
display(pd.read_csv(REV / 'paired_mae_bootstrap.csv'))

from feedback_checks import run as run_feedback
from verify_feedback import verify as verify_feedback
if RUN_FEEDBACK_CHECKS or RUN_TRAIN or RUN_REVISION_CHECKS:
    run_feedback()
print(json.dumps(verify_feedback(), indent=2))
display(pd.read_csv(OUT / 'feedback/matched_comparison.csv'))
display(pd.read_csv(OUT / 'feedback/decision_and_tail.csv'))
subprocess.run([sys.executable, 'report_source/render_review_figures.py'], check=True)
from IPython.display import Image as DisplayImage
for name in ['pipeline', 'timeline', 'hour', 'errors']:
    display(DisplayImage(filename=str(Path('report_source/figures') / (name + '.png'))))


## 16. Sources and package provenance

Source dataset: DOI 10.6084/m9.figshare.22495141.v1 (Lee, E.; Baek, K.; Kim, J., 2023).
Dataset paper: DOI 10.1038/s41597-024-02942-9 (Baek, K.; Lee, E.; Kim, J., 2024).
PACKAGE_INFO.json records input hashes, inherited modules and reused fitted model provenance.
No old report PDF is used as a figure or as executable evidence for the new manuscript.

In [ ]:
info=json.loads(Path('PACKAGE_INFO.json').read_text(encoding='utf-8'))
assert info['package_id']==PACKAGE_ID
assert sha256('data/ChargingRecords.xlsx')==info['inputs']['workbook']['sha256']
assert sha256('data/ChargingRecords_original.csv')==info['inputs']['canonical_csv']['sha256']
print('Package:',info['package_id'])
print('Workbook/CSV checks passed; historical model provenance is documented.')
print('Feature dictionary:',REV/'feature_dictionary_complete.csv')

## 17. Saved model and arrival-only inference

The model bundle includes fitted components, category vocabularies, metadata catalog and
completed development history. A new query supplies UserID, ChargerID and StartDatetime.
It must not supply the unknown current session's energy, duration or departure as predictors.
An optional feed of completed sessions enables online history. Without the feed, inference uses
frozen development history. Both identified and anonymous routes are demonstrated below.


In [ ]:
from predict_arrival import predict_sessions
example = pd.concat([test[test.known_user.eq(1)].head(2),
                     test[test.known_user.eq(0)].head(1)])[['ChargerID', 'UserID', 'StartDatetime']].copy()
display(predict_sessions(OUT / 'models/arrival_model_bundle.joblib', example))
print('No completed-session feed supplied: this demonstration uses frozen development history.')


## 18. Verification and limitations

Recompute metrics from saved predictions, reconstruct features and test saved-model inference.
The numerical comparison targets in the verification code refer to the historical study values;
the current Overleaf manuscript must be checked separately against the resulting tables.

Limits remain prior test exposure, one dataset, proxy reporting delay, assumed arrival metadata
availability and correlated history. The appended balanced and nested experiments address
selection comparisons within development windows; they do not make the final test untouched.
No historical report PDF or supplementary document is included as the current manuscript.


In [ ]:
check = subprocess.run([sys.executable, 'verify_results.py'], check=True, capture_output=True, text=True)
print(check.stdout)
print(json.dumps(verify_revision_outputs('.'), indent=2))
primary_checks = primary_report_checks('.')
revision_checks = rounded_report_checks(REV)
display(primary_checks)
display(revision_checks)
all_checks = pd.concat([primary_checks, revision_checks], ignore_index=True)
mismatches = all_checks.loc[~all_checks.matches.astype(bool)]
if len(mismatches):
    display(Markdown('**Report differences detected — retain computed results and inspect these rows.**'))
    display(mismatches)
else:
    print(f'PASS: all {len(all_checks)} checked full-study values (main report and supplementary results) agree within displayed rounding.')
print('Selected model:', result['selected_model'])
print('Identified-user online test:', result['test_metrics'])
print('New checks:', revision['report_checks_matching'], '/', revision['report_checks'])
archive_path = shutil.make_archive('Group25_analysis_outputs', 'zip', OUT)
print('Results-only archive:', archive_path)
print('Keep the full Colab package for data, modules, fitted models and notebook.')

print(json.dumps(verify_feedback(), indent=2))
from PIL import Image as PILImage
for folder in [OUT / 'figures', Path('report_source/figures')]:
    for path in folder.glob('*.png'):
        assert path.stat().st_size > 0, str(path)
        with PILImage.open(path) as im:
            im.verify()
print('All PNG artifacts are nonempty and decodable.')


## 20. Shared input and equal trial count — archive selection-score diagnostic

Six distinct configurations for each of Ridge, kNN, RF, HGB, LightGBM and CatBoost,
including each original setting, plus a fixed Linear/OLS reference. Five original
expanding temporal folds give **185 real fits**. All seven families share the exact
fold-fitted matrix, training rows and validation rows; kNN uses all fitting rows.

Default mode recomputes 740 fold metrics from saved validation predictions and
checks candidates, summaries, source/code hashes and unchanged primary artifacts.
Set `RUN_BALANCED_SEARCH=True` in Section 1 to repeat this diagnostic. Existing
results are preserved in a timestamped backup folder before a new run.

**Interpretation:** six configurations is an equal trial count, not equal runtime
or exhaustive coverage. The selected CV scores use the search folds and remain
exploratory; this is not nested CV or independent testing. No final-test scoring
or primary-model reselection is performed. The locked protocol lists all spaces,
candidates and matrix/row hashes.


In [ ]:
from datetime import datetime, timezone
from verify_balanced_search import verify as verify_balanced_search
BALANCED = OUT / 'balanced_search'
if RUN_BALANCED_SEARCH:
    if BALANCED.exists():
        backup = OUT / ('balanced_search_saved_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
        shutil.move(str(BALANCED), str(backup))
        print('Previous diagnostic preserved:', backup)
    subprocess.run([sys.executable, 'balanced_search.py', '--workers', '2'], check=True)
print(json.dumps(verify_balanced_search(), indent=2))
display(pd.read_csv(BALANCED / 'comparison.csv'))
display(pd.read_csv(BALANCED / 'selected_settings.csv'))


## 21. Retrospective nested temporal assessment — report Table 7

The same frozen 37 configurations are evaluated in **three inner expanding folds**
inside each of **five outer training sets**. Each inner partition fits its own encoder.
Both inner train and inner validation labels must be available before outer fitting;
training labels must also be available before the inner validation boundary.

Inner mean R² chooses one configuration per family and the family selector.
Only then are selected configurations refitted on outer training and scored on outer
validation: **555 inner + 35 regression outer + 5 Dummy fits = 595 actual fits**.
The selector reuses the chosen family's predictions; User mean is unfitted.

The default verifies saved predictions, exact partitions, inner-only choices and
unchanged primary artifacts. `RUN_NESTED_TEMPORAL=True` reruns this study with a
timestamped backup. Candidate design and these periods were explored earlier:
this is a retrospective check, not independent blind validation, and it does not
validate the original native-input CatBoost blend selection. No final test is scored.


In [ ]:
from verify_nested_temporal import verify as verify_nested_temporal
NESTED = OUT / 'nested_temporal'
if RUN_NESTED_TEMPORAL:
    if NESTED.exists():
        backup = OUT / ('nested_temporal_saved_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
        shutil.move(str(NESTED), str(backup))
        print('Previous nested assessment preserved:', backup)
    subprocess.run([sys.executable, 'nested_temporal.py', '--workers', '2'], check=True)
print(json.dumps(verify_nested_temporal(), indent=2))
display(pd.read_csv(NESTED / 'comparison.csv'))
display(pd.read_csv(NESTED / 'outer_metrics.csv')[['outer', 'model', 'trial', 'R2', 'MAE']])
from verify_inference_ids import verify as verify_inference_ids
from verify_inference_inputs import verify as verify_inference_inputs
verify_inference_ids()
verify_inference_inputs()
